In [3]:
!pip install matplotlib
import torch
import matplotlib.pyplot as plt
import numpy as np
import random
from torchvision import transforms
from src.masks.multiblock import MaskCollator as MBMaskCollator


mask_collator = MaskCollator(input_size=(224, 224), patch_size=16, nenc=3, npred=1)

# Function to create a random image (e.g., random noise image as placeholder for actual images)
def create_random_image(size=(224, 224)):
    return torch.rand(3, size[0], size[1])

# Create a batch of random images (3 images in the batch)
batch_size = 3
images = [create_random_image() for _ in range(batch_size)]

# Collate the images into a batch
collated_images = torch.stack(images)

# Generate masks for the batch
collated_batch, collated_masks_enc, collated_masks_pred = mask_collator(collated_images)

# Now, let's apply the masks to the images

def apply_mask_to_image(image, mask):
    """
    Applies a mask to an image by zeroing out the masked regions.
    """
    masked_image = image.clone()
    masked_image[mask == 0] = 0  # Zero out the masked regions
    return masked_image

# Apply the encoder and predictor masks to the images
masked_images_enc = []
masked_images_pred = []
for i in range(batch_size):
    image = collated_images[i]
    mask_enc = collated_masks_enc[i]
    mask_pred = collated_masks_pred[i]
    
    # For each mask in the list of masks (could be multiple for each image)
    masked_image_enc = apply_mask_to_image(image, mask_enc[0])  # Using first encoder mask for simplicity
    masked_image_pred = apply_mask_to_image(image, mask_pred[0])  # Using first predictor mask for simplicity
    
    masked_images_enc.append(masked_image_enc)
    masked_images_pred.append(masked_image_pred)

# Convert masked images to numpy for visualization
masked_images_enc_np = [img.permute(1, 2, 0).cpu().numpy() for img in masked_images_enc]
masked_images_pred_np = [img.permute(1, 2, 0).cpu().numpy() for img in masked_images_pred]

# Plot original images and their corresponding masked versions
fig, axes = plt.subplots(batch_size, 3, figsize=(12, batch_size * 4))

for i in range(batch_size):
    # Show original image
    axes[i, 0].imshow(collated_images[i].permute(1, 2, 0).cpu().numpy())
    axes[i, 0].set_title(f"Original Image {i+1}")
    axes[i, 0].axis('off')

    # Show encoder-masked image
    axes[i, 1].imshow(masked_images_enc_np[i])
    axes[i, 1].set_title(f"Masked Encoder {i+1}")
    axes[i, 1].axis('off')

    # Show predictor-masked image
    axes[i, 2].imshow(masked_images_pred_np[i])
    axes[i, 2].set_title(f"Masked Predictor {i+1}")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()



Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib]8 [matplotlib]


ModuleNotFoundError: No module named 'torchvision'